In [4]:
import pandas as pd
import difflib
import unicodedata
from rapidfuzz import process, utils

# 预处理：转小写、去除特殊字符、去除法语重音符号（如 é -> e）
def preprocess(text):
    return utils.default_process(text)

# 加载数据
official_df = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\Ecoles_primaires_2024_2025.xlsx', sheet_name=0)
missing_df = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\missing_project_schools_original.csv')
all_data_df = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final.csv')


# 对official_df相关字段0-1化:
##城市/乡村学校0-1处理
official_df = official_df[official_df['Milieu_ecole'].isin(['Rural', 'Urbain'])]
official_df = official_df.dropna(subset=['Milieu_ecole'])
Milieu_mapping = {
    'Rural': 0,
    'Urbain': 1
}
official_df['Milieu_ecole'] = official_df['Milieu_ecole'].replace(Milieu_mapping)
try:
    official_df['Milieu_ecole'] = official_df['Milieu_ecole'].astype(int)
except ValueError:
    print("警告：'Statut' 字段转换整数失败，可能存在未处理的非数值/非映射值。")
##公立/私立学校0-1处理
official_df = official_df[official_df['Statut'].isin(['Public', 'Privé'])]
official_df = official_df.dropna(subset=['Statut'])
Statut_mapping = {
    'Public': 0,
    'Privé': 1
}
official_df['Statut'] = official_df['Statut'].replace(Statut_mapping)
try:
    official_df['Statut'] = official_df['Statut'].astype(int)
except ValueError:
    print("警告：'Statut' 字段转换整数失败，可能存在未处理的非数值/非映射值。")


# 预处理匹配字段
official_df['norm_dren'] = official_df['DRENA'].apply(preprocess)
official_df['norm_ecole'] = official_df['Nom_ecole'].apply(preprocess)
missing_df['norm_dren'] = missing_df['Lib_DREN'].apply(preprocess)
missing_df['norm_ecole'] = missing_df['Ecole'].apply(preprocess)

# 执行模糊匹配，建立更正映射表
correction_map = {} # 格式: {(原区域, 原名): {'name': 新名, 'statut': 新性质, 'milieu': 新城乡}}

jilu=0
for _, row in missing_df.iterrows():
    orig_dren, orig_ecole = row['Lib_DREN'], row['Ecole']
    n_dren, n_ecole = row['norm_dren'], row['norm_ecole']
    
    # 缩小范围：仅在同一 DRENA 下寻找
    choices_subset = official_df[official_df['norm_dren'] == n_dren]
    
    if not choices_subset.empty:
        # 建立 规范化名字 -> {Nom_ecole, Statut, Milieu_ecole} 的字典
        #name_lookup = choices_subset.set_index('norm_ecole')[['Nom_ecole', 'Statut', 'Milieu_ecole']].to_dict('index')
        name_lookup = dict(zip(choices_subset['norm_ecole'], 
                               zip(choices_subset['Nom_ecole'], 
                                   choices_subset['Statut'], 
                                   choices_subset['Milieu_ecole'])))

        # 寻找最接近的字符串
        matches = difflib.get_close_matches(n_ecole, name_lookup.keys(), n=1, cutoff=0.7)   #######cutoff可更改@@@@
        
        if matches:
            correction_map[(orig_dren, orig_ecole)] = name_lookup[matches[0]]
            jilu=jilu+1

# 在 all_data_final 中应用更正
def apply_all_corrections(row):
    key = (row['Lib_DREN'], row['Ecole'])
    if key in correction_map:
        info = correction_map[key]
        row['Ecole'] = info[0]
        row['Statut'] = info[1]
        row['Milieu_ecole'] = info[2]
    return row
# 预处理：去除 Ecole 开头和结尾的空格
all_data_df['Ecole'] = all_data_df['Ecole'].str.strip()
all_data_df = all_data_df.apply(apply_all_corrections, axis=1)


# 保存结果
all_data_df.to_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_correction.csv', index=False)
print(f"缺失学校一共有1115条, 一共有{jilu}条学校成功更正Name、Statut、Milieu_ecole")
print("更正完成！结果已保存至 all_data_final_correction.csv")
print(correction_map)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26200\2809397510.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  official_df['Milieu_ecole'] = official_df['Milieu_ecole'].replace(Milieu_mapping)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_26200\2809397510.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  official_df['Statut'] = official_df['Statut'].replace(Statut_mapping)


缺失学校一共有1115条, 一共有973条学校成功更正Name、Statut、Milieu_ecole
更正完成！结果已保存至 all_data_final_correction.csv
{('BOUAKE 1', 'EPP ASSENGOU'): ('EPP ASSENGOU-PRI', 0, 0), ('SOUBRE', 'EPP 1B DE GRAND-ZATTRY'): ('EPP DE GRAND ZATTRY 3B', 0, 1), ('KORHOGO', 'EPP LOFINEKAHA'): ('EPP 2 DE LOFINEKAHA', 0, 0), ('BOUAFLE', 'EPP BONON LA PAIX 2'): ('EPP LA PAIX 1', 0, 0), ('SAN-PEDRO', 'EPP GNATO 1'): ('EPP GNATO 2', 0, 0), ('KORHOGO', 'EPP TIONKAHA'): ('EPP DE TIONKAHA', 0, 0), ('DUEKOUE', 'EPP TIESSAN'): ('EPP TIESSAN', 0, 0), ('KORHOGO', 'EPP LAGNONKAHA'): ('EPP DE LAGNONKAHA', 0, 1), ('GAGNOA', 'EPP BROZAN PAULKRO'): ('EPP BROUKRO', 0, 0), ('GAGNOA', 'EPP GOTA BAOULE 1'): ('EPP GOHUE 1', 0, 0), ('DUEKOUE', 'EPP DIEDROU-ZODRY'): ('EPP DIEDROU-ZODRY', 0, 1), ('KORHOGO', 'EPP SEKONKAHA'): ('EPP DE SEKONKAHA', 0, 0), ('KORHOGO', 'EPP DABOUNGO'): ('EPP DE DABOUNGO', 0, 0), ('DAOUKRO', 'EPP NANAN AKA KOUAKOU 1'): ('EPP 2 NANAN AKA KOUAKOU', 0, 1), ('GAGNOA', 'EPP GNAMAGNOA 1'): ('EPP GNATROA 1', 0, 0), ('AGBOVILLE

In [5]:
# 输出correction_map以检查正确性
comparison_list = []
for (dren, old_name), new_infom in correction_map.items():
    comparison_list.append({
        'DREN': dren,
        'Original_School': old_name,
        'Corrected_School': new_infom[0],
        'Corrected_Statut': new_infom[1],
        'Corrected_Milieu_ecole': new_infom[2],

    })

df_check = pd.DataFrame(comparison_list)
check_file_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\correction_verification_list.csv'
df_check.to_csv(check_file_path, index=False, encoding='utf-8-sig')

# 草稿
#### 手工处理 放弃

In [6]:
# import pandas as pd
# # 在merge之前，先对all_data_final表进行操作，
# # 一些参加了项目但匹配不到学校总表里的学校可能名字存在一些错误，查明并更正名字；
# df_project = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final.csv')
# replace_dict = {
#     'EPC  BAPTISTE AEEBCI KANOROBA': 'EPC BAPTISTE AEEBCI KANOROBA',
#     'EPC ASSEMBLEE DE DIEU DE GNIPI 2': 'EPC ASSEMBLEE DE DIEU',
#     'EPC ATTAZIB WATTAALIMI': 'EPC ATTAHZIB WATTALIM',
#     'EPC BAPTISTE AEBECI BETHANIE':'EPC BAPTISTE AEBECI BETHEL',
#     'EPC CATHOLIQUE NDA DUEKOUE':'EPC CATHOLIQUE 1 DUEKOUE',
#     'EPC DAR EL-HADISS':'EPC DAR EL-HADISS',
#     'EPC DAR FATWA':'EPC DAR FIRDAOUSS' 
#     # 添加更多替换规则
# }
# df_project['Ecole'] = df_project['Ecole'].replace(replace_dict)

# df_project.loc[(df_project['Ecole'] == 'EPC ADVENTISTE') & (df_project['Lib_DREN'] == 'BOUAKE 2'), 'Lib_DREN'] = 'BOUAKE 1'
# df_project.loc[(df_project['Ecole'] == 'EPC ANASS BOUN MALICK') & (df_project['Lib_DREN'] == 'BOUAKE 1'), 'Lib_DREN'] = 'BOUAKE 2'
# df_project.loc[(df_project['Ecole'] == 'EPC ASSEMBLEE DE DIEU') & (df_project['Lib_DREN'] == 'AGBOVILLE'), 'Lib_DREN'] = 'ABENGOUROU'
# df_project.loc[(df_project['Ecole'] == 'EPC CHEICK OUMAR') & (df_project['Lib_DREN'] == 'BOUAKE 2'), 'Lib_DREN'] = 'BOUAKE 1'

# df_project.to_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_correction.csv',index=False)